# 01 - The McCulloch-Pitts neuron

This is the very first artificial neuron, from 1943. It is simple enough to build with a handful of lines of plain Python, and yet it already contains the seed of every neural network that came after.

**What you need.** Basic Python (functions, lists, `if`, `for`). Nothing else. No math beyond adding and comparing numbers.

**What you will be able to do at the end.** Build a neuron by hand, make it behave like the logic gates inside a computer, draw what it is doing, see why one neuron is not enough for some problems, and fix that by connecting a few neurons together.

**How to use this notebook.** Run each cell with `Shift+Enter` and read the output before moving on. There are challenges along the way. Try each one before opening the solution, even if you only get halfway. That effort is where the learning happens.

## 1. The idea in one sentence

A neuron receives a few yes/no signals, counts how many are "yes", and if the count is big enough, it fires.

Let's make that concrete with a story. Imagine a committee of friends deciding whether to go to the cinema. Each friend says yes (1) or no (0). The rule is "we go if at least 2 people say yes". The committee is the neuron, the answers are the inputs, "at least 2" is the threshold, and "we go" is the output.

In [ ]:
friend_1 = 1
friend_2 = 0
friend_3 = 1

total_yes = friend_1 + friend_2 + friend_3
threshold = 2

we_go = total_yes >= threshold
print("Yes votes:", total_yes)
print("Do we go?", we_go)

Change the votes above (try `friend_3 = 0`) and run again. Then change `threshold` to 1, or to 3. Get a feel for it before we give it a name.

A **McCulloch-Pitts neuron** is exactly this committee. Warren McCulloch (a brain scientist) and Walter Pitts (a logician, only 20 years old) wrote it down in 1943 after looking at real neurons and keeping only two facts about them.

1. A neuron either fires or it does not. There is no "half firing". So the output is 1 or 0.
2. A neuron fires when enough signals arrive at the same time. So we count and compare with a threshold.

Everything else about real neurons (chemistry, timing, shape) was dropped on purpose. That is what a model is. Keep the part you care about, throw away the rest.

## 2. From story to function

Let's write the committee as a function so we can call it many times. The inputs are a list of 0s and 1s, the threshold is a number, the answer is 1 (fires) or 0 (stays quiet).

You will see `-> int` and `: list[int]` in the signature. Those are **type hints**. They do not change how the code runs, they tell the reader what kind of value goes in and comes out. We use them everywhere in these notebooks because they make code easier to read.

In [ ]:
def neuron(inputs: list[int], threshold: int) -> int:
    total = sum(inputs)
    if total >= threshold:
        return 1
    return 0


print(neuron([1, 0, 1], threshold=2))
print(neuron([1, 0, 0], threshold=2))
print(neuron([1, 0, 0], threshold=1))

Two yes votes with threshold 2 fires. One yes vote with threshold 2 does not. One yes vote with threshold 1 does.

Here is what the neuron looks like as a diagram. Inputs on the left, a circle that sums them and compares with the threshold, the output on the right. Every neuron picture you will ever see is a variation of this.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, FancyArrowPatch

INK = "#0b0b0b"
INK_SOFT = "#52514e"
GRID = "#e6e5e1"
ORANGE = "#eb6834"
BLUE = "#2a78d6"
SURFACE = "#fcfcfb"

plt.rcParams.update(
    {
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "axes.edgecolor": GRID,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.titleweight": "bold",
        "axes.titlelocation": "left",
        "legend.frameon": False,
    }
)

Point = tuple[float, float]


def draw_connection(ax: plt.Axes, start: Point, end: Point, veto: bool = False) -> None:
    if veto:
        ax.plot([start[0], end[0]], [start[1], end[1]], color=BLUE, lw=1.5)
        ax.scatter(*end, s=70, color=BLUE, zorder=4)
        return
    ax.add_patch(
        FancyArrowPatch(start, end, arrowstyle="-|>", color=INK, mutation_scale=14, lw=1.5)
    )


def draw_unit(ax: plt.Axes, center: Point, label: str, radius: float = 0.55) -> None:
    ax.add_patch(Circle(center, radius, fc="white", ec=INK, lw=1.5, zorder=3))
    ax.text(*center, label, ha="center", va="center", fontsize=9, zorder=5)


def draw_neuron_diagram(n_inputs: int, threshold: int, vetoes: list[bool] | None = None) -> None:
    vetoes = vetoes or [False] * n_inputs
    fig, ax = plt.subplots(figsize=(6, 1 + 0.8 * n_inputs))
    ys = [(n_inputs - 1) / 2 - i for i in range(n_inputs)]
    for i, y in enumerate(ys):
        label = f"x{i + 1}" + (" (veto)" if vetoes[i] else "")
        ax.text(-0.15, y, label, ha="right", va="center", fontsize=11)
        draw_connection(ax, (0, y), (1.65, 0), veto=vetoes[i])
    draw_unit(ax, (2.2, 0), f"sum >= {threshold}")
    draw_connection(ax, (2.75, 0), (3.8, 0))
    ax.text(3.95, 0, "y", ha="left", va="center", fontsize=11)
    ax.set(xlim=(-1.2, 4.4), ylim=(min(ys) - 0.8, max(ys) + 0.8))
    ax.set_aspect("equal")
    ax.axis("off")
    plt.show()


draw_neuron_diagram(n_inputs=3, threshold=2)

## 3. Two inputs, and a small discovery

From now on we use two inputs, `x1` and `x2`. Two is enough to see everything interesting, and it is easy to draw.

With two inputs there are only four possible situations. Let's list them all with threshold 2.

In [ ]:
for x1 in [0, 1]:
    for x2 in [0, 1]:
        output = neuron([x1, x2], threshold=2)
        print(f"x1={x1}  x2={x2}  ->  {output}")

The neuron fires only when both inputs are 1. In logic that is called **AND**.

Now change `threshold=2` to `threshold=1` in the cell above and run it again. Look at the result before reading on.

With threshold 1 the neuron fires when at least one input is 1. That is **OR**. Same neuron, one number changed, completely different logical rule. The threshold is the knob that decides what the neuron computes.

> ### Challenge: the other two thresholds
>
> Before running anything, predict the four outputs for `threshold=0` and for `threshold=3`. Then check with the loop.

<details>
<summary><b>Show solution</b></summary>

With threshold 0 the neuron **always fires**, because any sum is at least 0. With threshold 3 it **never fires**, because two inputs add up to at most 2.

```python
for t in [0, 3]:
    print(f"threshold {t}:", [neuron([x1, x2], t) for x1 in [0, 1] for x2 in [0, 1]])
```

So with two plain inputs a neuron can compute exactly four different things. Always, never, AND, OR. There are 16 possible yes/no functions of two inputs, so we are far from covering them all. Keep that number in mind.

</details>

## 4. Truth tables

We are going to look at that four-row table many times, so let's write a helper that prints it. The helper takes a function as an argument. If that looks odd, it just means "give me something I can call with `x1, x2` and I will call it on every combination".

In [ ]:
from collections.abc import Callable

Gate = Callable[[int, int], int]


def show_table(gate: Gate, name: str) -> None:
    print(f"x1  x2  |  {name}")
    print("--------+------")
    for x1 in [0, 1]:
        for x2 in [0, 1]:
            print(f" {x1}   {x2}  |  {gate(x1, x2)}")


def and_gate(x1: int, x2: int) -> int:
    return neuron([x1, x2], threshold=2)


def or_gate(x1: int, x2: int) -> int:
    return neuron([x1, x2], threshold=1)


show_table(and_gate, "AND")
print()
show_table(or_gate, "OR")

A table like this is called a **truth table**. For a function with a handful of yes/no inputs, the truth table is a complete description. Two gates with the same truth table are the same gate, no matter how they are built inside. We will use that fact to check our work.

## 5. Saying "no", the inhibitory input

So far every input pushes the neuron toward firing. Real neurons also have inputs that do the opposite. McCulloch and Pitts included this with a strong rule. If an **inhibitory** input is 1, the neuron does not fire, full stop. It is a veto.

In [ ]:
draw_neuron_diagram(n_inputs=2, threshold=1, vetoes=[False, True])

In [ ]:
def neuron_with_veto(inputs: list[int], vetoes: list[bool], threshold: int) -> int:
    for value, is_veto in zip(inputs, vetoes, strict=True):
        if is_veto and value == 1:
            return 0
    votes = [value for value, is_veto in zip(inputs, vetoes, strict=True) if not is_veto]
    if sum(votes) >= threshold:
        return 1
    return 0

Read it top to bottom. If any veto input is on, return 0 right away. Otherwise collect the normal votes, count them, and compare with the threshold as before.

Now a neat trick. A neuron with **only** a veto input and threshold 0. With no normal votes the sum is 0, and 0 is at least 0, so it fires by default. The veto switches it off. That is **NOT**.

In [ ]:
def not_gate(x: int) -> int:
    return neuron_with_veto([x], vetoes=[True], threshold=0)


print("NOT 0 =", not_gate(0))
print("NOT 1 =", not_gate(1))

We now have AND, OR and NOT. This matters more than it looks. Every digital circuit in your computer is built from those three gates. So if one neuron can do each of them, a large enough group of neurons can compute anything a computer can. That was the main claim of the 1943 paper.

One more gate we will need later. One normal input, one veto input, threshold 1. It fires when `x1` is on **and** `x2` is off. The diagram above is exactly this neuron.

In [ ]:
def x1_and_not_x2(x1: int, x2: int) -> int:
    return neuron_with_veto([x1, x2], vetoes=[False, True], threshold=1)


show_table(x1_and_not_x2, "x1 AND NOT x2")

> ### Challenge: NOR and NAND
>
> NOR fires only when both inputs are off. NAND fires unless both inputs are on. Build both. One of them is possible with a single `neuron_with_veto`, the other is not. Which is which, and why?

<details>
<summary><b>Show solution</b></summary>

**NOR** is one neuron. Two vetoes, threshold 0. It fires by default and either input switches it off.

```python
def nor_gate(x1: int, x2: int) -> int:
    return neuron_with_veto([x1, x2], vetoes=[True, True], threshold=0)
```

**NAND** cannot be a single neuron. It needs to fire at `(1,0)` and `(0,1)`, so neither input can be a veto. But then both are normal votes, and the only options are the four from Challenge 1. None of those is NAND. So we build it from two neurons.

```python
def nand_gate(x1: int, x2: int) -> int:
    return not_gate(and_gate(x1, x2))
```

The lesson is that the veto rule is very rigid. Later models replace it with negative weights, and NAND becomes a single neuron again.

</details>

## 6. The step function

Look at the comparison `sum(votes) >= threshold`. Move the threshold to the other side.

    sum(votes) - threshold >= 0

Same rule, rearranged. Now the neuron reads as "compute a number, then fire if it is not negative". That shape is how every modern neuron is written. The number is called the **pre-activation** and usually named `z`. The "fire if not negative" part is a function of its own, the **step function**.

In [ ]:
def step(z: float) -> int:
    if z >= 0:
        return 1
    return 0


zs = [z / 10 for z in range(-30, 31)]
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(zs, [step(z) for z in zs], color=ORANGE, lw=2.5)
ax.axvline(0, color=GRID, lw=1)
ax.set(
    xlabel="z = sum of votes - threshold", ylabel="output", yticks=[0, 1], title="The step function"
)
ax.grid(True, color=GRID)
plt.show()

Flat at 0, then jumps to 1 exactly at zero, then flat at 1. Nothing in between. Keep this picture in mind. Much later, when you meet sigmoid and ReLU, they are smoother replacements for this jump, chosen so that learning algorithms can work with them.

Here is `neuron` rewritten with `step`. Compare it with section 2. Same behaviour, new shape.

In [ ]:
def neuron_v2(inputs: list[int], threshold: int) -> int:
    z = sum(inputs) - threshold
    return step(z)


print(neuron_v2([1, 1], threshold=2), neuron_v2([1, 0], threshold=2))

## 7. Drawing what a neuron does

Numbers in a table are fine, but a picture shows something the table hides.

Put `x1` on the horizontal axis and `x2` on the vertical. The four inputs become four points. Colour each point by the output. Orange fires, blue does not. Then draw the line `x1 + x2 = threshold` (nudged by half a step so it sits between the points instead of on them).

In [ ]:
def draw_points(ax: plt.Axes, gate: Gate, title: str) -> None:
    for x1 in [0, 1]:
        for x2 in [0, 1]:
            colour = ORANGE if gate(x1, x2) == 1 else BLUE
            ax.scatter(x1, x2, color=colour, s=220, edgecolor=INK, zorder=3)
    ax.set(
        xlim=(-0.5, 1.5),
        ylim=(-0.5, 1.5),
        xticks=[0, 1],
        yticks=[0, 1],
        xlabel="x1",
        ylabel="x2",
        title=title,
    )
    ax.set_aspect("equal")
    ax.grid(True, color=GRID)


def draw_threshold_line(ax: plt.Axes, threshold: int) -> None:
    xs = [-0.5, 1.5]
    line = [threshold - 0.5 - x for x in xs]
    ax.plot(xs, line, color=INK, ls="--", lw=1.5)
    ax.fill_between(xs, line, 1.5, color=ORANGE, alpha=0.08, zorder=0)


fig, axes = plt.subplots(1, 2, figsize=(8, 4))
draw_points(axes[0], and_gate, "AND, line at x1 + x2 = 1.5")
draw_threshold_line(axes[0], threshold=2)
draw_points(axes[1], or_gate, "OR, line at x1 + x2 = 0.5")
draw_threshold_line(axes[1], threshold=1)
fig.tight_layout()
plt.show()

In both pictures one straight line separates orange from blue. That is no accident. The rule `x1 + x2 >= threshold` describes a half-plane. Everything above the line fires, everything below does not.

What does changing the threshold do to the line? Let's draw all four thresholds side by side.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3.6))
for ax, t in zip(axes, [0, 1, 2, 3], strict=True):
    draw_points(ax, lambda x1, x2, t=t: neuron([x1, x2], t), f"threshold = {t}")
    draw_threshold_line(ax, t)
fig.tight_layout()
plt.show()

The line **slides** diagonally as the threshold grows. It never rotates and it never bends, because all inputs count equally. That is the full range of a single McCulloch-Pitts neuron with two inputs. Slide one fixed diagonal line and say "this side yes, that side no".

Problems that one straight line can solve are called **linearly separable**. Remember the phrase, it comes back in a moment.

> ### Challenge: read the line from the picture
>
> Without running code, look at the `threshold = 3` panel. Where is the line, and why does it produce the output it does?

<details>
<summary><b>Show solution</b></summary>

The line is at `x1 + x2 = 2.5`, above and to the right of every point. All four points are below it, so all four are blue. The neuron never fires.

Similarly at `threshold = 0` the line is at `x1 + x2 = -0.5`, below and to the left of everything. All points are above it, so all fire.

The general rule is that the line `x1 + x2 = threshold - 0.5` has the points with sum `>= threshold` on the upper side.

</details>

## 8. The problem a neuron cannot solve

Here is a fourth gate. **XOR**, "exclusive or". It fires when the inputs are different and stays quiet when they are the same.

Let's draw it before building it.

In [ ]:
def xor_target(x1: int, x2: int) -> int:
    return 1 if x1 != x2 else 0


fig, ax = plt.subplots(figsize=(4, 4))
draw_points(ax, xor_target, "XOR, fires when inputs differ")
plt.show()

Try to draw one straight line that separates orange from blue. Take your time.

You cannot. The orange points sit on one diagonal, the blue points on the other. Any line ends up with an orange and a blue point on the same side.

Let's stop trusting the picture and check every threshold against the XOR truth table.

In [ ]:
def same_table(gate_a: Gate, gate_b: Gate) -> bool:
    return all(gate_a(x1, x2) == gate_b(x1, x2) for x1 in [0, 1] for x2 in [0, 1])


for threshold in [0, 1, 2, 3]:
    candidate: Gate = lambda x1, x2, t=threshold: neuron([x1, x2], t)  # noqa: E731
    print(f"threshold {threshold}: same as XOR? {same_table(candidate, xor_target)}")

> ### Challenge: vetoes do not help either
>
> Extend the search. For each threshold in 0 to 3 and each way of marking the two inputs as veto or not (four ways), check whether the neuron matches XOR. How many combinations did you test, and how many matched?

<details>
<summary><b>Show solution</b></summary>

```python
from itertools import product

count = 0
for threshold, vetoes in product([0, 1, 2, 3], product([False, True], repeat=2)):
    candidate = lambda x1, x2, t=threshold, v=list(vetoes): neuron_with_veto([x1, x2], v, t)
    count += 1
    if same_table(candidate, xor_target):
        print("found:", threshold, vetoes)
print("tested", count, "neurons")
```

16 neurons tested, zero matches. There is no single McCulloch-Pitts neuron for XOR, and now we know it by exhaustion, not just by looking at a picture.

</details>

This limitation was made famous in 1969 by a book called *Perceptrons*, and it scared many researchers away from neural networks for about fifteen years. Which is a little sad, because the fix was already in the 1943 paper.

## 9. The fix, connect neurons together

Say XOR in words. "At least one input is on, and not both."

We have a neuron for "at least one is on" (OR). We have one for "both are on" (AND). And we have one for "the first thing and not the second thing" (`x1_and_not_x2`). So we build XOR in two steps.

1. Compute `h1 = OR(x1, x2)` and `h2 = AND(x1, x2)`.
2. Feed `h1` and `h2` into `x1_and_not_x2`.

The neurons in step 1 form a **hidden layer** (hidden because you never see their values from outside). The neuron in step 2 is the **output layer**.

In [ ]:
def draw_xor_network() -> None:
    fig, ax = plt.subplots(figsize=(8, 4))
    inputs: dict[str, Point] = {"x1": (0, 1), "x2": (0, -1)}
    hidden: dict[str, Point] = {"OR\nsum >= 1": (2.2, 1), "AND\nsum >= 2": (2.2, -1)}
    out: Point = (4.6, 0)
    for name, pos in inputs.items():
        ax.text(*pos, name, ha="center", va="center", fontsize=12)
        for target in hidden.values():
            draw_connection(ax, (pos[0] + 0.3, pos[1]), (target[0] - 0.6, target[1]))
    for name, pos in hidden.items():
        draw_unit(ax, pos, name)
    draw_connection(ax, (2.75, 1), (3.95, 0.35))
    draw_connection(ax, (2.75, -1), (3.95, -0.35), veto=True)
    draw_unit(ax, out, "h1 AND\nNOT h2\nsum >= 1", radius=0.7)
    draw_connection(ax, (5.3, 0), (6.1, 0))
    ax.text(6.25, 0, "y", ha="left", va="center", fontsize=12)
    ax.text(2.2, 2.1, "hidden layer", ha="center", fontsize=10, color=INK_SOFT)
    ax.text(4.6, 2.1, "output layer", ha="center", fontsize=10, color=INK_SOFT)
    ax.set(xlim=(-0.6, 6.8), ylim=(-2, 2.6))
    ax.set_aspect("equal")
    ax.axis("off")
    plt.show()


draw_xor_network()

Black arrows are normal inputs. The blue line ending in a dot is the veto (that dot is the standard way to draw an inhibitory connection in neuroscience). Now the code, which is shorter than the drawing.

In [ ]:
def xor_network(x1: int, x2: int) -> int:
    h1 = or_gate(x1, x2)
    h2 = and_gate(x1, x2)
    return x1_and_not_x2(h1, h2)


show_table(xor_network, "XOR")
print()
print("Same as the target?", same_table(xor_network, xor_target))

It works. And the output neuron is literally the same `x1_and_not_x2` function from section 5. It has no idea its inputs are now the outputs of other neurons. That indifference is what makes layers stackable, and it is the same reason a modern framework lets you chain layers without each one knowing what came before.

## 10. Why it works, in a picture

This is the most important picture in the notebook.

Instead of plotting the original inputs `(x1, x2)`, plot the hidden layer's outputs `(h1, h2)` for each of the four cases, keeping the XOR colours. Left panel is what the network receives, right panel is what the output neuron receives.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4.2))
draw_points(axes[0], xor_target, "What the network sees, (x1, x2)")

ax = axes[1]
for x1 in [0, 1]:
    for x2 in [0, 1]:
        h1, h2 = or_gate(x1, x2), and_gate(x1, x2)
        colour = ORANGE if xor_target(x1, x2) == 1 else BLUE
        ax.scatter(h1, h2, color=colour, s=220, edgecolor=INK, zorder=3)
        ax.annotate(
            f"from ({x1},{x2})",
            (h1, h2),
            xytext=(10, -14 if x2 else 8),
            textcoords="offset points",
            fontsize=8,
        )
xs = [0.0, 1.5]
line = [x - 0.5 for x in xs]
ax.plot(xs, line, color=INK, ls="--", lw=1.5)
ax.fill_between(xs, -0.5, line, color=ORANGE, alpha=0.08, zorder=0)
ax.set(xlim=(-0.5, 1.5), ylim=(-0.5, 1.5), xticks=[0, 1], yticks=[0, 1])
ax.set(
    xlabel="h1 = OR(x1, x2)",
    ylabel="h2 = AND(x1, x2)",
    title="What the output neuron sees, (h1, h2)",
)
ax.set_aspect("equal")
ax.grid(True, color=GRID)
fig.tight_layout()
plt.show()

Follow the points. `(0,0)` stayed at `(0,0)`. Both orange points, `(0,1)` and `(1,0)`, landed on the **same** spot, `(1,0)`. And `(1,1)` moved to `(1,1)`. Three spots instead of four, and now a single line separates orange from blue with room to spare.

The hidden layer did not solve XOR. It **moved the points** so that XOR became a problem one neuron can solve. That is the whole trick of deep learning in one sentence. Each layer re-arranges the data so the next layer's straight-line decision becomes possible. Everything you will learn later (weights, training, backpropagation) is about finding good re-arrangements automatically, instead of designing them by hand like we just did.

> ### Challenge: XOR a second way
>
> XOR can also be said as "(x1 and not x2) or (x2 and not x1)". Build a network for that phrasing. How many neurons does it use, and what does its hidden space look like?

<details>
<summary><b>Show solution</b></summary>

```python
def x2_and_not_x1(x1: int, x2: int) -> int:
    return neuron_with_veto([x1, x2], vetoes=[True, False], threshold=1)


def xor_network_b(x1: int, x2: int) -> int:
    h1 = x1_and_not_x2(x1, x2)
    h2 = x2_and_not_x1(x1, x2)
    return or_gate(h1, h2)


print(same_table(xor_network_b, xor_target))
```

Three neurons again, two hidden and one output. In hidden space the two blue points `(0,0)` and `(1,1)` both land on `(0,0)`, while the orange points go to `(1,0)` and `(0,1)`. Then OR separates them. Different re-arrangement, same effect. There is usually more than one hidden layer that works.

</details>

## 11. The same neuron in NumPy

So far we used plain lists and loops so nothing was hidden. Real networks handle thousands of inputs and millions of examples, and Python loops are far too slow for that. The standard tool is **NumPy**, which lets you write an operation once and apply it to a whole table of examples at the same time.

Here is the neuron in NumPy. Read it next to `neuron_v2` from section 6.

In [ ]:
import numpy as np
from numpy.typing import NDArray

Binary = NDArray[np.int_]


def np_step(z: NDArray[np.number]) -> Binary:
    return (z >= 0).astype(np.int_)


def np_neuron(x: Binary, threshold: int) -> Binary:
    z = x.sum(axis=1) - threshold
    return np_step(z)


all_four = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])

print("AND:", np_neuron(all_four, threshold=2))
print("OR: ", np_neuron(all_four, threshold=1))

`x` is a table with one row per example and one column per input. `x.sum(axis=1)` adds up each row. Subtract the threshold, apply the step, and you get all four answers in one call with no `for`. The `Binary` name is a type hint meaning "a NumPy array of integers", so signatures stay readable.

From the next notebook on we use NumPy everywhere. If it feels unfamiliar, come back and compare the two versions until it clicks.

> ### Challenge: XOR network in NumPy
>
> Rewrite `xor_network` so it takes the `all_four` table and returns all four XOR outputs at once. You will need a NumPy version of the veto neuron.

<details>
<summary><b>Show solution</b></summary>

```python
Mask = NDArray[np.bool_]


def np_neuron_with_veto(x: Binary, vetoes: Mask, threshold: int) -> Binary:
    vetoed = x[:, vetoes].any(axis=1)
    z = x[:, ~vetoes].sum(axis=1) - threshold
    return np.where(vetoed, 0, np_step(z))


def np_xor_network(x: Binary) -> Binary:
    h1 = np_neuron(x, threshold=1)
    h2 = np_neuron(x, threshold=2)
    hidden = np.column_stack([h1, h2])
    return np_neuron_with_veto(hidden, vetoes=np.array([False, True]), threshold=1)


print(np_xor_network(all_four))  # [0 1 1 0]
```

`np.column_stack` glues the two hidden outputs into a new table with one column per hidden neuron. That table is the hidden space from section 10, as an array.

</details>

## 12. Where this is going

Put the 1943 neuron next to the neuron used today.

| | McCulloch-Pitts (1943) | Today |
|---|---|---|
| Inputs | 0 or 1 | any number |
| How much each input counts | all the same, or a veto | a separate **weight** per input, can be negative |
| Threshold | a fixed integer you pick | a **bias**, learned |
| Fire or not | step function | smooth functions (sigmoid, ReLU) |
| How it is set up | by hand | learned from examples |

The shape is the same. Add up the inputs, shift by something, decide. What is missing is **learning**. We picked every threshold by hand here. The next model, Rosenblatt's perceptron (1958), keeps this exact neuron and adds a rule that adjusts the weights on its own whenever it makes a mistake.

## Exercises

1. Build a three-input neuron that fires when at least two of the three inputs are on (a majority vote). Print all eight cases.
2. Build a network that fires when **exactly one** of two inputs is on... wait, that is XOR. Build one that fires when exactly one of **three** inputs is on. You will need a hidden layer.
3. Section 3 found that two plain inputs give only 4 of the 16 possible functions. Extend the count. With vetoes allowed, how many of the 16 can a single neuron reach? (Enumerate all 16 neuron configurations, collect the distinct truth tables, count them.)
4. Draw the hidden space for your solution to exercise 2. Is it linearly separable?

## Further reading

- Michael Nielsen, *Neural Networks and Deep Learning*, chapter 1. Free online, gentle, starts right here.
- The original paper, McCulloch and Pitts (1943), *A Logical Calculus of the Ideas Immanent in Nervous Activity*. Read the introduction, skip the logic notation.

**Next notebook.** The perceptron. Same neuron, plus learning.